# 01 - LiteLLM Basics

All LLM access in Atlas goes through the **LiteLLM gateway** — a single OpenAI-compatible front door for every provider (containerized Ollama, host Ollama, OpenAI, Anthropic, OpenRouter, …). This notebook walks through chat, streaming, and embeddings using the standard `openai` SDK pointed at LiteLLM.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# OPENAI_API_BASE / OPENAI_API_KEY are set by the JupyterHub startup
# script and resolve to the LiteLLM gateway. The vanilla openai SDK
# works unchanged.
client = OpenAI(
    base_url=os.environ["OPENAI_API_BASE"],
    api_key=os.environ["OPENAI_API_KEY"],
)

## 1. List available models

In [ ]:
# Models are registered in volumes/litellm/config.yaml. Identifiers are
# of the form <provider>/<model-name> — e.g. `ollama/qwen3.6:latest`,
# `gpt-4o`, `claude-sonnet-4-6`.
models = client.models.list()
print("Available models:")
for m in models.data:
    print(f"  • {m.id}")

## 2. Simple chat

In [ ]:
# Pick any model from the list above. Default is the seeded Ollama
# content model.
model = os.getenv("LITELLM_DEFAULT_MODEL", "ollama/qwen3.6:latest")

response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}],
)
print(response.choices[0].message.content)

## 3. Streaming responses

In [ ]:
stream = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Write a haiku about AI."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="", flush=True)
print()

## 4. Generate embeddings

Embeddings go through LiteLLM's `/v1/embeddings`. Use any embedding model registered in `volumes/litellm/config.yaml` — e.g. `ollama/nomic-embed-text` or `openai/text-embedding-3-small`.

In [ ]:
embedding_model = os.getenv("LITELLM_EMBEDDING_MODEL", "ollama/nomic-embed-text")

result = client.embeddings.create(
    model=embedding_model,
    input="Atlas",
)
vec = result.data[0].embedding
print(f"Embedding dimension: {len(vec)}")
print(f"First 5 values: {vec[:5]}")